# Explore Experiment Results

This notebook loads and visualizes results from the theory-driven experiment.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add parent to path
sys.path.insert(0, str(Path.cwd().parent))

from analysis import paired_bootstrap, check_refutation_criteria

%matplotlib inline
sns.set_style('whitegrid')

## Load Results

In [ ]:
results_dir = Path("../results")

# Load all result JSON files
results = []
for f in results_dir.glob("**/*.json"):
    if f.name == "stage_0_gate.json":  # Skip gate files
        continue
    with open(f) as fp:
        results.append(json.load(fp))

print(f"Loaded {len(results)} result files")

# Convert to DataFrame
df = pd.DataFrame(results)
df.head()

## Results Summary

In [ ]:
# Summary by arm and budget
summary = df.groupby(['benchmark', 'arm', 'budget']).agg({
    'test_mean': ['mean', 'std', 'count'],
    'runtime_seconds': 'mean',
}).round(3)

print("Results Summary:")
print(summary)

## Budget Usage

In [ ]:
# Extract budget stats
budget_data = []
for r in results:
    if 'budget_stats' in r:
        budget_data.append({
            'arm': r['arm'],
            'budget': r['budget'],
            'total_units': r['budget_stats']['total_units'],
            'total_cost': r['budget_stats']['total_cost'],
            'task_calls': r['budget_stats']['task_lm_calls'],
            'reflection_calls': r['budget_stats']['reflection_lm_calls'],
        })

budget_df = pd.DataFrame(budget_data)

# Plot budget usage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Units used vs cap
budget_summary = budget_df.groupby(['arm', 'budget']).mean()
budget_summary[['total_units']].unstack().plot(kind='bar', ax=axes[0])
axes[0].set_title('Total Units Used by Arm and Budget')
axes[0].set_ylabel('Units')
axes[0].legend(title='Budget')

# Cost
budget_summary[['total_cost']].unstack().plot(kind='bar', ax=axes[1])
axes[1].set_title('Total Cost by Arm and Budget')
axes[1].set_ylabel('Cost ($)')
axes[1].legend(title='Budget')

plt.tight_layout()
plt.show()

## Performance Comparison

In [ ]:
# Plot test scores by arm
fig, ax = plt.subplots(figsize=(12, 6))

# Filter for primary benchmark
hover_df = df[df['benchmark'] == 'hover']

# Plot by budget
for budget in sorted(hover_df['budget'].unique()):
    subset = hover_df[hover_df['budget'] == budget]
    means = subset.groupby('arm')['test_mean'].mean()
    stds = subset.groupby('arm')['test_mean'].std()
    
    x = range(len(means))
    ax.errorbar(x, means, yerr=stds, label=f'Budget {budget}', 
                marker='o', capsize=5, capthick=2)

ax.set_xticks(range(len(means)))
ax.set_xticklabels(means.index)
ax.set_xlabel('Arm')
ax.set_ylabel('Test Score')
ax.set_title('Performance by Arm and Budget (HoVer)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Paired Comparisons

In [ ]:
# Compare K-Jev vs GEPA default (Arm A)
hover_df = df[df['benchmark'] == 'hover']

# Get scores for budget 50 and 100
k_scores = hover_df[
    (hover_df['arm'] == 'K-Jev') & 
    (hover_df['budget'].isin([50, 100]))
]['test_mean'].tolist()

a_scores = hover_df[
    (hover_df['arm'] == 'A') & 
    (hover_df['budget'].isin([50, 100]))
]['test_mean'].tolist()

if k_scores and a_scores:
    result = paired_bootstrap(k_scores, a_scores, n_samples=10000)
    
    print("K-Jev vs A (pooled B∈{50,100}):")
    print(f"  Mean difference: {result['mean_difference']:.3f}")
    print(f"  95% CI: [{result['ci_lower']:.3f}, {result['ci_upper']:.3f}]")
    print(f"  P-value: {result['p_value']:.4f}")
    print(f"  Significant: {result['significant']}")
else:
    print("Not enough data for paired comparison")

## Refutation Criteria

Check all pre-registered refutation criteria (R1-R7, H_cal).

In [ ]:
# Organize results by arm
results_by_arm = {}
for r in results:
    arm = r['arm']
    if arm not in results_by_arm:
        results_by_arm[arm] = []
    results_by_arm[arm].append(r)

# Check criteria
criteria = check_refutation_criteria(results_by_arm)

print("\nRefutation Criteria:")
print("="*60)
for key, value in criteria.items():
    print(f"\n{key}:")
    for k, v in value.items():
        print(f"  {k}: {v}")

## Export for Paper

Save summary tables for inclusion in paper/report.

In [ ]:
# Main results table
summary.to_csv('../results/summary_table.csv')
print("Saved summary_table.csv")

# Refutation criteria
with open('../results/refutation_criteria.json', 'w') as f:
    json.dump(criteria, f, indent=2)
print("Saved refutation_criteria.json")